In [ ]:
import actionet
import numpy as np
import anndata
import pandas as pd

import lets_plot
lets_plot.LetsPlot.setup_html(no_js=True)

In [ ]:
adata = anndata.read_h5ad("../data/adata_agg_Scn4b_OX_fil_post_lazy.h5ad", backed="r")
adata

In [ ]:
# actionet.normalize_anndata(adata, layer=None, log_base=2, inplace=True)
lzt = actionet.create_lazy_transform(adata, target_sum=10000, log_base=2, pseudocount=1)

In [ ]:
# adata.X[1:10, 1:10].todense()

In [ ]:
markers = actionet.find_markers(adata, adata.obs['CellType'], features_use="Gene", top_genes=30, return_type='dataframe', lazy_transform=lzt)

In [ ]:
markers

In [ ]:
annots_out = actionet.annotate_cells(adata, markers, layer = None, method='vision', features_use='Gene', network_key='actionet', use_enrichment=True, use_lpa=False, n_threads=0, lazy_transform=lzt)

In [ ]:
adata.obs['annot'] = annots_out['labels']
adata.obs['annot_conf'] = annots_out['confidence']
adata.obsm['annot_enrichment'] = annots_out['enrichment']

In [ ]:
actionet.plot_umap(adata, color='annot', trans_attr="annot_conf")

In [ ]:
actionet.plot_umap(adata, color='annot', trans_attr="annot_conf")

In [ ]:
annots_clust = actionet.annotate_clusters(adata, markers=markers, cluster_key="leiden", features_use="Gene", layer="logcounts")

In [ ]:
annots_clust

In [ ]:
cluster_to_annotation = dict(zip(annots_clust['cluster_names'], annots_clust['labels']))
adata.obs['annot_leiden'] = adata.obs['leiden'].map(cluster_to_annotation)

In [ ]:
actionet.plot_umap(adata, color='annot_leiden')

In [ ]:
# adata.write_h5ad("../data/test_adata_post_annot.h5ada")

In [ ]:
markers.to_csv("../data/test_markers.csv", sep=',', header=True, index=False)